In [1]:
import pandas as pd
import numpy as np
import math
import re
from pathlib import Path
from urllib.parse import urlparse

# Set display options to see all columns
pd.set_option('display.max_columns', None)
print("Environment Ready.")

Environment Ready.


In [2]:
df = pd.read_csv(Path.cwd().parent /'master_dataset_clean.csv')

df.head()

,url,label
0,noxlogic.nl,0
1,dubaiexch.live,0
2,brightroulettegroup.com,0
3,worldglobalmedia.co.uk,0
4,fobo-friends.ru,0


In [3]:
def get_entropy(text):
    """Calculates the Shannon Entropy (randomness) of a string."""
    if not text: 
        return 0
    # Calculate frequency of each character
    probabilities = [n_x/len(text) for n_x in pd.Series(list(text)).value_counts()]
    # Calculate Shannon Entropy
    entropy = -sum(p * math.log2(p) for p in probabilities)
    return entropy

def is_ip(url):
    """Checks if the domain/hostname is a raw IP address."""
    # Regex pattern to match IPv4, IPv4 in hex, and IPv6
    match = re.search(
        '(([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.'
        '([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\/)|'  
        '((0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\/)' 
        '(?:[a-fA-F0-9]{1,4}:){7}[a-fA-F0-9]{1,4}', url)
    return 1 if match else 0

print("Helper functions defined.")

Helper functions defined.


In [4]:
def extract_lexical_features(url):
    """Breaks down a URL into numerical features."""
    features = {}
    url_str = str(url)
    
    # Parse the URL components
    if not url.startswith('http://') and not url.startswith('https://'):
        url = 'http://' + url
    parsed = urlparse(url)
    hostname = parsed.netloc
    path = parsed.path
    
    # --- Structural Lengths ---
    features['url_length'] = len(url_str)
    features['hostname_length'] = len(hostname)
    
    # --- Character Counts (The Fingerprint) ---
    features['dot_count'] = url_str.count('.')
    features['hyphen_count'] = url_str.count('-')
    features['at_count'] = url_str.count('@')
    features['slash_count'] = url_str.count('/')
    features['query_count'] = url_str.count('?')
    
    # --- Security Indicators ---
    features['is_ip'] = is_ip(url_str)
    features['url_entropy'] = get_entropy(url_str)
    
    return features

print("Extraction engine ready.")

Extraction engine ready.


In [5]:
print("Extracting features... Please wait.")

df['url'] = df['url'].astype(str)

# Wrap in try/except to skip malformed URLs like bad IPv6
def safe_extract(url):
    try:
        return extract_lexical_features(url)
    except Exception:
        # Return zeros for any unparseable URL
        return {
            'url_length': 0, 'hostname_length': 0, 
            'dot_count': 0, 'hyphen_count': 0, 'at_count': 0,
            'slash_count': 0, 'query_count': 0, 
            'is_ip': 0, 'url_entropy': 0
        }

feature_list = df['url'].apply(safe_extract).tolist()

feature_df = pd.DataFrame(feature_list)
df_final = pd.concat([df, feature_df], axis=1)

# Drop rows where all features are 0 (malformed URLs)
df_final = df_final[df_final['url_length'] > 0]

print(f"Extraction complete. Rows after cleaning: {len(df_final)}")
df_final.head()

Extracting features... Please wait.
Extraction complete. Rows after cleaning: 585034


,url,label,url_length,hostname_length,dot_count,hyphen_count,at_count,slash_count,query_count,is_ip,url_entropy
0,noxlogic.nl,0,11,11,1,0,0,0,0,0,2.913977
1,dubaiexch.live,0,14,14,1,0,0,0,0,0,3.521641
2,brightroulettegroup.com,0,23,23,1,0,0,0,0,0,3.642490
3,worldglobalmedia.co.uk,0,22,22,2,0,0,0,0,0,3.754442
4,fobo-friends.ru,0,15,15,1,1,0,0,0,0,3.506891


In [6]:
# Save to CSV for the Model Training phase
df_final.to_csv(Path.cwd().parent /'master_dataset_lexical_features.csv', index=False)

print("Success! File saved as 'master_dataset_lexical_features.csv'.")
print(f"Final Column List: {df_final.columns.tolist()}")

Success! File saved as 'master_dataset_lexical_features.csv'.
Final Column List: ['url', 'label', 'url_length', 'hostname_length', 'dot_count', 'hyphen_count', 'at_count', 'slash_count', 'query_count', 'is_ip', 'url_entropy']


In [7]:
# CHECKING

# After extraction, check what safe URLs look like
safe_sample = df_final[df_final['label'] == 0].head(5)
print("Safe URL features:")
print(safe_sample[['url', 'slash_count',  'hostname_length']].to_string())

phish_sample = df_final[df_final['label'] == 1].head(5)
print("\nPhishing URL features:")
print(phish_sample[['url', 'slash_count', 'hostname_length']].to_string())

Safe URL features:
                       url  slash_count  hostname_length
0              noxlogic.nl            0               11
1           dubaiexch.live            0               14
2  brightroulettegroup.com            0               23
3   worldglobalmedia.co.uk            0               22
4          fobo-friends.ru            0               15

Phishing URL features:
                                                                                      url  slash_count  hostname_length
300000                                                             112.93.202.244:48456/i            1               20
300001                                                              221.1.227.205:35291/i            1               19
300002  binary-block-tabel-expert-get.wiki/92c764c1-ff82-4023-a702-309971bc7633/google.ct            2               34
300003                                                        112.93.202.244:48456/bin.sh            1               20
300004  binary-